# US Wildfire Dashboard: An Interactive Panel Tutorial

## Dataset

This dashboard uses the [US Wildfire Records, 6th Edition](https://www.kaggle.com/datasets/behroozsohrabi/us-wildfire-records-6th-edition) dataset, which contains over 2.3 million wildfire records reported across the United States from 1992 through 2020. Each record includes information such as the year the fire occurred, its cause classification, fire size, geographic location, and containment dates.

I chose this dataset because recent Canadian wildfires have caused poor air quality across much of the United States, including my area, which sparked my interest in learning more about wildfire patterns. The dataset also lends itself well to interactive visualization because it contains temporal, categorical, geographic, and continuous variables that can be explored together. This makes it possible to build multiple coordinated visualizations that allow users to examine wildfire activity from different perspectives.

Because the original dataset is very large, it was cleaned and transformed into two smaller datasets that are used by this dashboard. The complete preprocessing workflow is documented in `01_data_preparation.ipynb`.

## Visualization Technique

This dashboard uses four types of charts to explore different aspects of US wildfires from 1992–2020. The visualizations group fires into three cause categories: **Human**, **Natural**, and **Undetermined**. Each category is assigned a consistent color throughout the dashboard so users can easily compare the same group across multiple charts.

### Line Chart

The line chart shows how the frequency of wildfires has changed over time, broken down by cause. It allows users to compare long-term trends and observe how the number of fires from each cause has varied throughout the years.

### Bar Chart

Unlike the line chart, which measures *how many* fires occurred, the bar chart measures *how much damage* each cause category produced by displaying the total acres burned. This chart is intentionally static, serving as a constant reference while the other visualizations allow users to interactively filter the data.

### Scatter Plot

The scatter plot explores the relationship between wildfire size and the number of days required to contain each fire. Fire size is displayed on a logarithmic scale because the data is highly right-skewed. Without a log scale, a relatively small number of extremely large fires would compress the much more common small fires into a narrow region of the plot, making patterns difficult to observe.

### Point Map

The point map displays the geographic locations of individual wildfires using a random sample of records. A point map was chosen instead of a choropleth because aggregating fires to the state level would hide important spatial patterns. For example, wildfire activity along California's coast versus its inland regions would disappear if the entire state were represented by a single color.

## How the Visualizations Complement Each Other

Each visualization answers a different question about the data. The line chart shows **when** wildfires occurred, the bar chart summarizes **how much damage** each cause produced, the scatter plot explores **how fire size relates to containment time**, and the point map reveals **where** wildfires occurred.

The line and bar charts summarize the entire dataset using aggregated statistics, while the scatter plot and point map display individual wildfire records. Together, these visualizations provide both high-level summaries and detailed observations, allowing users to explore the dataset from multiple perspectives.

## Interactivity

The dashboard includes two interactive controls that are shared across multiple visualizations. However, each chart only responds to the controls that support its intended purpose.

- The **Cause Category** selector (multi-select) updates the line chart, scatter plot, and point map, allowing users to compare or isolate specific wildfire causes.
- The **Year Range** slider updates only the scatter plot and point map. It is intentionally omitted from the line chart because restricting the year range would reduce its effectiveness at displaying long-term trends.
- The bar chart does not respond to either control, allowing it to remain a stable reference that always displays the complete dataset.

## Visualization Library

The framework I chose to build my dashboard is Panel. Panel is an open-source Python library, originally developed by HoloViz, that provides tools for building interactive dashboards, web applications, and exploratory data analysis interfaces. It can be installed using pip install panel or conda install panel.

I chose Panel for several reasons. First, it integrates naturally with Jupyter notebooks, making it easy to develop and test interactive dashboards within the notebook environment. Every chart in this notebook was built and refined interactively before being incorporated into the final dashboard.

Second, Panel provides a clean way to organize interactive applications by combining widgets, layouts, and visualizations into a single dashboard. The dashboard uses Panel's FastListTemplate to create a consistent layout with a sidebar for filters and a main area for the visualizations. The dashboard can be run locally using panel serve and deployed as a web application using a hosting service, allowing others to access it through a web browser without installing Python themselves.

There are some limitations to this approach. Because Panel applications run on a Python server, they require a hosting service when deployed publicly. In addition, rendering many individual observations or performing computationally expensive operations can reduce responsiveness. To improve performance, the original wildfire dataset was preprocessed by aggregating the data used in the line and bar charts and using a random sample for the scatter plot and geographic point map.

The final reason I chose Panel is that it is plotting library agnostic. It works with many different Python visualization libraries rather than locking you into one. This flexibility allowed me to pair it with hvPlot, another HoloViz library built on HoloViews, which provides a simple, high-level interface for creating interactive visualizations directly from pandas DataFrames. It can be installed with pip install hvplot or conda install hvplot and integrates seamlessly with Panel, making it a natural fit for this project.

Panel and hvPlot are both declarative rather than procedural. Instead of writing step-by-step instructions for how a chart should be drawn, you describe what the chart should show—which columns to use, which chart type to display, and how the data should be encoded—and the library handles the rendering. This also extends to interactivity. Each chart is written once as a function that depends on specific widget values, and Panel automatically updates the visualization whenever those values change, rather than requiring manually written callback logic for every possible interaction.



## Step 1: Static Visualizations

Before building each chart, the required libraries and processed data files are loaded.

In [ ]:
# Import the libraries needed to build our dashboard
import panel as pn
import hvplot.pandas
import pandas as pd
import numpy as np
import param

pn.extension()

wildfire_agg = pd.read_csv('data/wildfire_agg.csv')
wildfire_sample = pd.read_csv('data/wildfire_sample.csv')

In [ ]:
cause_colors = {
    'Human': '#f1ce63',
    'Natural': '#4e79a7',
    'Undetermined': '#e15759',
}

### Fires per Year

An early version of the line chart plotted all 13 detailed cause categories at once. With that many lines, two colors were nearly identical, and most causes were crushed into an unreadable band near zero.

![13-cause line chart, hard to read](images/line_chart_13_causes.png)

The 13 categories were grouped into three broader ones, Human, Natural, and Undetermined, using the dataset's own `NWCG_CAUSE_CLASSIFICATION` field. This immediately made the chart readable, and revealed something the detailed view had obscured. "Undetermined" looked like the single largest cause in the 13-category breakdown, but once grouped, Human-caused fires clearly dominate in frequency.

In [ ]:
line_data = wildfire_agg.groupby(['FIRE_YEAR', 'CAUSE'])['fire_count'].sum().reset_index()

line_chart = line_data.hvplot.line(
    x='FIRE_YEAR', y='fire_count', by='CAUSE',
    color=[cause_colors[c] for c in line_data['CAUSE'].unique()],
    width=700, height=400,
    xlabel='Year', ylabel='Number of Fires',
    title='Wildfires per Year by Cause Category',
    yformatter='%.0f',
    hover_cols=['FIRE_YEAR', 'fire_count', 'CAUSE']
).opts(
    tools=['hover'],
    hover_tooltips=[('Year', '@FIRE_YEAR'), ('Fires', '@fire_count{0,0}'), ('Category', '@CAUSE')]
)
line_chart

### Total Acres Burned by Cause

An earlier version of this chart plotted all 13 detailed cause categories, and the same magnitude problem showed up here as it did on the line chart: a couple of categories dominated so heavily that the rest were nearly invisible. 

![Total acres burned by 13 individual causes, linear scale](images/bar_chart_13_causes_linear.png)

A log scale was tried to make the smaller bars visible, but it flattened the true scale of the difference, which was the more important story to tell.

![Total acres burned by 13 individual causes, log scale](images/bar_chart_13_causes_log_scale.png)

While this made every category visible, it flattened the true scale of the difference between them, which was the more important story to tell.

The real fix was the same one used for the line chart: grouping the 13 causes into three broader categories, Human, Natural, and Undetermined. With only three bars instead of thirteen, the true magnitude difference between categories could be shown clearly on a linear scale, with plain number formatting, no log scale needed.

In [ ]:
bar_totals = wildfire_agg.groupby('CAUSE')['total_acres'].sum().reset_index().sort_values('total_acres', ascending=False)

bar_chart = bar_totals.hvplot.bar(
    x='CAUSE', y='total_acres',
    color='CAUSE', cmap=cause_colors,
    width=500, height=400,
    xlabel='Cause Category', ylabel='Total Acres Burned',
    title='Total Acres Burned by Cause Category (1992–2020)',
    yformatter='%.0f',
    legend=False,
    hover_cols=['CAUSE', 'total_acres']
).opts(
    tools=['hover'],
    hover_tooltips=[('Category', '@CAUSE'), ('Total Acres', '@total_acres{0,0}')]
)
bar_chart

### Fire Size vs. Days to Containment

In [ ]:
scatter_data = wildfire_sample.dropna(subset=['DAYS_TO_CONTAIN'])
scatter_data = scatter_data[scatter_data['DAYS_TO_CONTAIN'] >= 0]

scatter_chart = scatter_data.hvplot.scatter(
    x='DAYS_TO_CONTAIN', y='FIRE_SIZE',
    by='CAUSE',
    color=[cause_colors[c] for c in scatter_data['CAUSE'].unique()],
    logy=True, alpha=0.35, size=25,
    width=700, height=400,
    xlabel='Days to Containment', ylabel='Fire Size (acres, log scale)',
    title='Fire Size vs. Days to Containment, by Cause Category',
    yformatter='%.0f',
    hover_cols=['CAUSE', 'FIRE_SIZE', 'DAYS_TO_CONTAIN']
).opts(
    tools=['hover'],
    hover_tooltips=[('Category', '@CAUSE'), ('Fire Size (acres)', '@FIRE_SIZE{0,0}'), ('Days to Containment', '@DAYS_TO_CONTAIN')]
)
scatter_chart

Fire size and containment duration are explored here, colored by cause. A large share of fires are contained the same day they're discovered, creating a dense cluster at zero days. This is a real pattern in the data, since most fires are small and quickly controlled, not an error.

### Wildfire Locations

An early version of the map included fires from Alaska and Puerto Rico, pushing the zoom out so far that the continental US, where most of the data and this dashboard's story live, shrank to a barely visible cluster.

![Map with extreme zoom](images/map_incorrect.png)

Restricting the map to continental US coordinates (latitude 24 to 50, longitude negative 125 to negative 66) resolved the issues.

In [ ]:
continental_sample = wildfire_sample[
    (wildfire_sample['LATITUDE'].between(24, 50)) &
    (wildfire_sample['LONGITUDE'].between(-125, -66))
]

map_chart = continental_sample.hvplot.points(
    x='LONGITUDE', y='LATITUDE', geo=True, tiles='OSM',
    color='CAUSE', cmap=cause_colors,
    alpha=0.5, size=6,
    width=700, height=450,
    xlabel='Longitude', ylabel='Latitude',
    title='Wildfire Locations by Cause Category (Continental US)',
    hover_cols=['CAUSE', 'STATE', 'FIRE_YEAR']
).opts(
    tools=['hover'],
    hover_tooltips=[('Category', '@CAUSE'), ('State', '@STATE'), ('Year', '@FIRE_YEAR')]
)
map_chart

## Step 2: Adding Interactivity

In [ ]:
CHART_WIDTH = 560
CHART_HEIGHT = 380

class WildfireDashboard(param.Parameterized):
    year_range = param.Range(default=(1992, 2020), bounds=(1992, 2020))
    cause_groups = param.ListSelector(
        default=['Human', 'Natural', 'Undetermined'],
        objects=['Human', 'Natural', 'Undetermined']
    )
 
    @param.depends('cause_groups')
    def line_chart(self):
        selected = self.cause_groups if self.cause_groups else ['Human', 'Natural', 'Undetermined']
        data = wildfire_agg[wildfire_agg['CAUSE'].isin(selected)]
        data = data.groupby(['FIRE_YEAR', 'CAUSE'])['fire_count'].sum().reset_index()
        return data.hvplot.line(
            x='FIRE_YEAR', y='fire_count', by='CAUSE',
            color=[cause_colors[c] for c in selected],
            width=CHART_WIDTH, height=CHART_HEIGHT,
            xlabel='Year', ylabel='Number of Fires',
            title='Wildfires per Year by Cause Category',
            ylim=(0, 100000), yformatter='%.0f',
            hover_cols=['FIRE_YEAR', 'fire_count', 'CAUSE']
        ).opts(
            tools=['hover'],
            hover_tooltips=[('Year', '@FIRE_YEAR'), ('Fires', '@fire_count{0,0}'), ('Category', '@CAUSE')]
        )
 
    @param.depends()
    def bar_chart(self):
        totals = wildfire_agg.groupby('CAUSE')['total_acres'].sum().reset_index().sort_values(
            'total_acres', ascending=False
        )
        return totals.hvplot.bar(
            x='CAUSE', y='total_acres',
            color='CAUSE', cmap=cause_colors,
            width=CHART_WIDTH, height=CHART_HEIGHT,
            xlabel='Cause Category', ylabel='Total Acres Burned',
            title='Total Acres Burned by Cause Category (1992–2020)',
            yformatter='%.0f', legend=False,
            hover_cols=['CAUSE', 'total_acres']
        ).opts(
            tools=['hover'],
            hover_tooltips=[('Category', '@CAUSE'), ('Total Acres', '@total_acres{0,0}')]
        )
 
    @param.depends('year_range', 'cause_groups')
    def scatter_chart(self):
        selected = self.cause_groups if self.cause_groups else ['Human', 'Natural', 'Undetermined']
        data = wildfire_sample[
            (wildfire_sample['FIRE_YEAR'] >= self.year_range[0]) &
            (wildfire_sample['FIRE_YEAR'] <= self.year_range[1]) &
            (wildfire_sample['CAUSE'].isin(selected))
        ]
        data = data.dropna(subset=['DAYS_TO_CONTAIN'])
        data = data[data['DAYS_TO_CONTAIN'] >= 0]
        return data.hvplot.scatter(
            x='DAYS_TO_CONTAIN', y='FIRE_SIZE',
            by='CAUSE',
            color=[cause_colors[c] for c in selected],
            logy=True, alpha=0.35, size=25,
            width=CHART_WIDTH, height=CHART_HEIGHT,
            xlabel='Days to Containment', ylabel='Fire Size (acres, log scale)',
            title='Fire Size vs. Days to Containment',
            yformatter='%.0f',
            hover_cols=['CAUSE', 'FIRE_SIZE', 'DAYS_TO_CONTAIN']
        ).opts(
            tools=['hover'],
            hover_tooltips=[('Category', '@CAUSE'), ('Fire Size (acres)', '@FIRE_SIZE{0,0}'), ('Days to Containment', '@DAYS_TO_CONTAIN')]
        )
 
    @param.depends('year_range', 'cause_groups')
    def map_chart(self):
        selected = self.cause_groups if self.cause_groups else ['Human', 'Natural', 'Undetermined']
        data = wildfire_sample[
            (wildfire_sample['FIRE_YEAR'] >= self.year_range[0]) &
            (wildfire_sample['FIRE_YEAR'] <= self.year_range[1]) &
            (wildfire_sample['CAUSE'].isin(selected)) &
            (wildfire_sample['LATITUDE'].between(24, 50)) &
            (wildfire_sample['LONGITUDE'].between(-125, -66))
        ]
        return data.hvplot.points(
            x='LONGITUDE', y='LATITUDE', geo=True, tiles='OSM',
            color='CAUSE',
            cmap=cause_colors,
            alpha=0.5, size=6,
            width=CHART_WIDTH, height=CHART_HEIGHT,
            xlabel='Longitude', ylabel='Latitude',
            title='Wildfire Locations (Continental US)',
            hover_cols=['CAUSE', 'STATE', 'FIRE_YEAR']
        ).opts(
            tools=['hover'],
            hover_tooltips=[('Category', '@CAUSE'), ('State', '@STATE'), ('Year', '@FIRE_YEAR')]
        )
 
 
dashboard = WildfireDashboard()
 
# ── Widgets (explicit types, sized to fit the sidebar) ──────
cause_widget = pn.widgets.MultiChoice.from_param(
    dashboard.param.cause_groups,
    name='Cause Category',
    width=230
)
year_widget = pn.widgets.RangeSlider.from_param(
    dashboard.param.year_range,
    name='Year Range',
    width=230
)
 
# ── Layout ─────────────────────────────────────────────────
layout = pn.template.FastListTemplate(
    title="US Wildfire Dashboard",
    sidebar_width=260,
    sidebar=[
        pn.pane.Markdown("### Filters"),
        cause_widget,
        year_widget
    ],
    main=[
        pn.Row(dashboard.line_chart, dashboard.bar_chart),
        pn.Row(dashboard.scatter_chart, dashboard.map_chart)
    ]
)
 
layout.servable()

## Step 3: The Interactive Dashboard in Action

Screenshots below demonstrate the dashboard's default state and how the
two controls affect it.

### Default View

All three cause categories are selected by default, and the year range
spans the full dataset. The bar chart is identical here and in every
other screenshot, since it doesn't respond to either control.

![Dashboard default view, all causes selected](images/dashboard_default_view.png)

### Filtering by Cause

Selecting a single cause, Natural, updates the line, scatter, and map
charts, while the bar chart stays unchanged.

![Dashboard filtered to Natural-caused fires only](images/dashboard_cause_filtered.png)

### Filtering by Year Range

Narrowing the year range to 2000–2010, with Human and Natural selected, updates the scatter and map charts. The line chart does not respond to the year range and continues to show the complete 1992–2020 trend.

![Dashboard filtered to Human and Natural causes, year range 2000-2010](images/dashboard_year_filtered.png)

### Bar Chart Detail

A closer look at the bar chart shows plain axis numbers instead of scientific notation, and consistent coloring with the other three charts.

![Total acres burned by cause, close-up](images/dashboard_bar_chart.png)

### Map Detail

Zooming into the map shows individual fire locations color-coded by cause, here centered on Michigan.

![Wildfire locations zoomed to Michigan](images/dashboard_map_working.png)

## Troubleshooting

**Choosing a deployment method.** The initial plan was to convert the dashboard into a standalone HTML file using Panel's Pyodide support, so it could run entirely in the browser with no server needed. This worked for the line, bar, and scatter charts, but the map, which uses `geoviews` for its base map tiles, caused the page to load indefinitely and never finish. Removing the map's tile background resolved the hang, but since keeping the tiled map was preferred, a different deployment approach was needed
instead.

Hugging Face Spaces was tried next, since it runs a real Python server rather than converting the app to run in the browser. The Docker option, needed to install `geoviews` properly, required a paid tier on the account used for this project, so this option was set aside as well.

The dashboard is instead deployed on Render, a free hosting service that runs the app as a standard Python web service rather than converting it. This avoids the Pyodide conversion issue entirely, since `geoviews` and the tiled map work the same way they do locally. One limitation of the free tier is that the service spins down after a period of inactivity, so the first visit after a while may take 30 to 60 seconds to load while the server restarts.

## Deploy Instructions

This dashboard is deployed on [Render](https://render.com), a free hosting service that runs the app as a standard Python web service. Deployment was chosen over Panel's browser-based Pyodide conversion because the map's tiled background depends on `geoviews`, which does not convert reliably to run in the browser (see Troubleshooting).

**To deploy this dashboard from scratch:**

1. Push the project to a GitHub repository, including `dashboard.py`, `requirements.txt`, and the `data/` folder with `wildfire_agg.csv` and `wildfire_sample.csv`.

2. Create a free account at [render.com](https://render.com).

3. From the Render dashboard, select **New → Web Service** and connect the GitHub repository.

4. Configure the service with the following settings:
   
   - **Runtime:** Python 3
   
   - **Build Command:** pip install -r requirements.txt

   - **Start Command:** panel serve dashboard.py --address 0.0.0.0 --port $PORT --allow-websocket-origin='*'

   - **Instance Type:** Free

5. Click **Create Web Service**. Render installs the dependencies and starts the app, which typically takes a few minutes on the first deploy.

6. Once the build finishes, the live dashboard is available at the URL Render provides.

**Note:** on the free tier, the service spins down after a period of inactivity. The first visit after a period of inactivity may take 30 to 60 seconds to load while the server restarts.

**Live dashboard:** [Click here](https://wildfire-dashboard-xpkc.onrender.com/dashboard)